# 🏆 Fantasy Copilot — Agente Analista con IA

> Pipeline de extracción y análisis automático de contenido de YouTube para recomendaciones de alineación en LaLiga Fantasy, basado en RAG y modelos de lenguaje.

## ⚙️ Bloque 1 — Instalación de librerías

Instalamos todas las dependencias necesarias para el proyecto.

| Librería | Uso |
|---|---|
| `youtube-transcript-api` | Descarga de transcripciones de YouTube |
| `openai` | Extracción de insights y embeddings |
| `chromadb` | Base de datos vectorial para el RAG |
| `google-api-python-client` | YouTube Data API v3 |
| `gradio` | Interfaz de chatbot |
| `tiktoken` | Cálculo de tokens y costes |

In [1]:
!pip install youtube-transcript-api==1.2.4 -q
!pip install openai -q
!pip install chromadb -q
!pip install google-api-python-client -q
!pip install gradio -q
!pip install tiktoken -q

print("✅ Librerías instaladas")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 817.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

## 🔑 Bloque 2 — Imports y configuración

Importamos todas las librerías necesarias e inicializamos los clientes de OpenAI y YouTube.

Las API keys se introducen en runtime mediante `getpass`. Nunca quedan expuestas en el código.

In [2]:
import os
import json
import re
from getpass import getpass
from datetime import datetime
from typing import Optional

from openai import OpenAI
import tiktoken
from youtube_transcript_api import YouTubeTranscriptApi
from googleapiclient.discovery import build
import gradio as gr

# --- API KEYS ---
OPENAI_API_KEY  = getpass("🔑 Introduce tu OpenAI API Key: ")
YOUTUBE_API_KEY = getpass("🔑 Introduce tu YouTube API Key: ")

# Inicializar cliente OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

# Verificar conexión
test = client.chat.completions.create(
    model="gpt-5.4-nano",
    max_completion_tokens=20,
    messages=[{"role": "user", "content": "Di Hola en español de España"}]
)
print("✅ OpenAI conectado:", test.choices[0].message.content)
print("✅ Imports completados")

🔑 Introduce tu OpenAI API Key: ··········
🔑 Introduce tu YouTube API Key: ··········
✅ OpenAI conectado: Parece que quieres decir **“Di hola”** en español de España. En
✅ Imports completados


## 📺 Bloque 3 — Obtención de videos de YouTube

Conectamos con la YouTube Data API para obtener los videos de análisis fantasy de los canales registrados.

**Decisiones de diseño:**
- Los channel IDs se hardcodean en `CANALES_FANTASY` para evitar llamadas innecesarias a la API
- Cada canal tiene sus propias `keywords` para filtrar videos relevantes
- Se excluyen videos del Mundial y videos sin número de jornada identificable
- Los videos se filtran por jornada objetivo (`JORNADA_OBJETIVO = 38`)
- Los resultados se ordenan por fecha descendente. El video más reciente de cada jornada tiene más peso

**Canales registrados:**
- 📺 Carrasco
- 📺 Jornada Perfecta Fantasy

> Para añadir un nuevo canal en el futuro, basta con añadir una entrada al diccionario `CANALES_FANTASY` con su `channel_id` y `keywords`.

In [3]:
youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)

CANALES_FANTASY = {
    "carrasco": {
        "channel_id": "UCu7ZOzCUgWpvvJzYAdnw0_Q",
        "keywords": ["jornada"]
    },
    "jornada_perfecta": {
        "channel_id": "UCvMw41zGXyYYPDvyXcHZfug",
        "keywords": ["jornada", " j3", " j4"]  # Cubre "J38", "J37", etc. con espacio previo para evitar falsos positivos
    },
}

def obtener_uploads_playlist_id(channel_id: str) -> str:
    response = youtube.channels().list(
        id=channel_id,
        part="contentDetails"
    ).execute()
    return response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

def es_video_relevante(titulo: str, keywords: list) -> bool:
    titulo_lower = titulo.lower()

    # Filtros de exclusión
    if "mundial" in titulo_lower:
        return False

    # Debe contener alguna keyword
    if not any(kw.lower() in titulo_lower for kw in keywords):
        return False

    # Debe tener número de jornada identificable (elimina los J?)
    if extraer_numero_jornada(titulo) is None:
        return False

    return True

def extraer_numero_jornada(titulo: str) -> Optional[int]:
    """Extrae el número de jornada del título. Soporta 'Jornada 38' y 'J38'."""
    match = re.search(r'(?:jornada\s*|j)(\d+)', titulo.lower())
    return int(match.group(1)) if match else None

def obtener_videos_canal(canal_key: str, max_videos: int = 50, solo_relevantes: bool = True, jornada_filtro: Optional[int] = None) -> list:
    if canal_key not in CANALES_FANTASY:
        raise ValueError(f"Canal '{canal_key}' no registrado. Disponibles: {list(CANALES_FANTASY.keys())}")

    canal_info = CANALES_FANTASY[canal_key]
    channel_id = canal_info["channel_id"]
    keywords   = canal_info["keywords"]

    playlist_id = obtener_uploads_playlist_id(channel_id)
    response = youtube.playlistItems().list(
        playlistId=playlist_id,
        part="id,snippet",
        maxResults=max_videos
    ).execute()

    videos = []
    for item in response.get("items", []):
        titulo   = item["snippet"]["title"]
        video_id = item["snippet"]["resourceId"]["videoId"]

        if solo_relevantes and not es_video_relevante(titulo, keywords):
            continue

        jornada = extraer_numero_jornada(titulo)

        # Filtrar por jornada específica si se indica
        if jornada_filtro is not None and jornada != jornada_filtro:
            continue

        videos.append({
            "canal":    canal_key,
            "video_id": video_id,
            "titulo":   titulo,
            "fecha":    item["snippet"]["publishedAt"][:10],
            "jornada":  jornada,
            "url":      f"https://youtube.com/watch?v={video_id}"
        })

    videos.sort(key=lambda v: (v["jornada"] or 0, v["fecha"]), reverse=True)
    return videos


# --- TEST ambos canales, solo J38 ---
JORNADA_OBJETIVO = 38

for canal_key in CANALES_FANTASY.keys():
    print(f"\n📹 Videos J{JORNADA_OBJETIVO} de [{canal_key}]:\n")
    videos = obtener_videos_canal(canal_key, max_videos=50, solo_relevantes=True, jornada_filtro=JORNADA_OBJETIVO)

    if not videos:
        print(f"  ⚠️  Sin videos de J{JORNADA_OBJETIVO}.")
    else:
        for v in videos:
            jornada_str = f"J{v['jornada']}" if v['jornada'] else "J?"
            print(f"  [{jornada_str}] [{v['fecha']}] {v['titulo']}")


📹 Videos J38 de [carrasco]:

  [J38] [2026-05-23] LA PREVIA FANTASY DE LA JORNADA 38 CON TODAS LAS ALINEACIONES CONFIRMADAS
  [J38] [2026-05-23] 🔝 TOP 10 CAPITANES para la JORNADA 38 en LaLiga Fantasy 25/26
  [J38] [2026-05-23] MIS TIER LIST FANTASY POR POSICIONES PARA LA JORNADA 38
  [J38] [2026-05-22] Mis ALINEACIONES PROBABLES y PRONÓSTICOS para la jornada 38 con TODA la ÚLTIMA HORA
  [J38] [2026-05-22] 🚀 3 JUGADORES que harán más de 10 PUNTOS en la JORNADA 38 de LaLiga Fantasy 25/26
  [J38] [2026-05-21] El MEJOR EQUIPO para la JORNADA 38 de LaLiga Fantasy SOLO con EQUIPOS con OBJETIVOS
  [J38] [2026-05-21] 🔒 Los 7 CLAUSULAZOS Uber Eats para la JORNADA 38 de LaLiga Fantasy
  [J38] [2026-05-21] ⭐️ MI ALINEACIÓN DE CHOLLOS FANTASY PARA LA JORNADA 38
  [J38] [2026-05-21] EL NOTICIERO FANTASY | MI NUEVO PROYECTO | TRIPLES PARA LA JORNADA 38
  [J38] [2026-05-20] Los MEJORES JUGADORES para la JORNADA 38 de LaLiga Fantasy 25/26
  [J38] [2026-05-19] ANÁLISIS FANTASY de la JORNADA 38 con AL

## 📝 Bloque 4 — Extracción de transcripciones

Descargamos las transcripciones en crudo de cada video usando `youtube-transcript-api`.

**Problema conocido:** YouTube bloquea peticiones desde IPs de servidores cloud como Google Colab. Se soluciona usando un proxy residencial de [Webshare](https://www.webshare.io).

**Flujo:**
1. Si `transcripciones_j38.json` existe en disco → carga directamente sin llamar a YouTube
2. Si no existe → descarga todas las transcripciones a través del proxy y guarda el JSON

> 💾 **Archivo generado:** `transcripciones_j38.json`. Descárgalo y consérvalo entre sesiones para evitar repetir las llamadas a YouTube.

**Credenciales necesarias:**
- Proxy Username y Proxy Password de Webshare (plan gratuito es suficiente)

In [4]:
import os

if os.path.exists("transcripciones_j38.json"):
    print("📂 Cargando transcripciones desde JSON local...")
    with open("transcripciones_j38.json", "r", encoding="utf-8") as f:
        videos_con_transcripcion = json.load(f)
    ok    = sum(1 for v in videos_con_transcripcion if v["estado"] == "ok")
    error = sum(1 for v in videos_con_transcripcion if v["estado"] != "ok")
    print(f"✅ {len(videos_con_transcripcion)} videos cargados — {ok} ✅ con transcripción — {error} ❌ sin transcripción")

else:
    print("⚠️  JSON no encontrado.")

📂 Cargando transcripciones desde JSON local...
✅ 17 videos cargados — 17 ✅ con transcripción — 0 ❌ sin transcripción


In [7]:
from youtube_transcript_api.proxies import WebshareProxyConfig

WEBSHARE_USERNAME = getpass("🔑 Introduce tu Webshare Proxy Username: ")
WEBSHARE_PASSWORD = getpass("🔑 Introduce tu Webshare Proxy Password: ")

🔑 Introduce tu Webshare Proxy Username: ··········
🔑 Introduce tu Webshare Proxy Password: ··········
📹 Total videos a procesar: 17

🧪 Test con primer video...

❌ Proxy no funciona: error: VideoUnplayable: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=A
Revisa tus credenciales de Webshare antes de continuar.


In [ ]:
# Test directo de credenciales Webshare
from youtube_transcript_api.proxies import WebshareProxyConfig

ytt_test = YouTubeTranscriptApi(
    proxy_config=WebshareProxyConfig(
        proxy_username=WEBSHARE_USERNAME,
        proxy_password=WEBSHARE_PASSWORD,
    )
)

try:
    fragmentos = ytt_test.fetch("AT_UTIeNmtI", languages=["es"])
    texto = " ".join(f.text for f in fragmentos)
    print(f"✅ Funciona: {len(texto):,} caracteres")
    print(texto[:200])
except Exception as e:
    print(f"❌ Error completo: {type(e).__name__}")
    print(str(e))

In [11]:
ytt_api = YouTubeTranscriptApi(
    proxy_config=WebshareProxyConfig(
        proxy_username=WEBSHARE_USERNAME,
        proxy_password=WEBSHARE_PASSWORD,
    )
)

def extraer_transcripcion(video: dict) -> dict:
    """Descarga la transcripción en crudo de un video."""
    video_id = video["video_id"]
    try:
        fragmentos  = ytt_api.fetch(video_id, languages=["es", "en"])
        texto_crudo = " ".join(fragmento.text for fragmento in fragmentos)
        return {**video, "transcripcion_cruda": texto_crudo, "estado": "ok"}
    except Exception as e:
        return {**video, "transcripcion_cruda": None, "estado": f"error: {type(e).__name__}: {str(e)[:80]}"}


def extraer_todas_transcripciones(videos: list) -> list:
    """Itera todos los videos y extrae sus transcripciones."""
    resultados = []
    total      = len(videos)

    for i, video in enumerate(videos, 1):
        print(f"  [{i}/{total}] {video['canal']} — {video['titulo'][:60]}...")
        resultado = extraer_transcripcion(video)

        if resultado["estado"] == "ok":
            chars = len(resultado["transcripcion_cruda"])
            print(f"           ✅ {chars:,} caracteres")
        else:
            print(f"           ❌ {resultado['estado']}")

        resultados.append(resultado)

    return resultados


# --- Obtener todos los videos ---
todos_videos = []
for canal_key in CANALES_FANTASY.keys():
    videos = obtener_videos_canal(canal_key, max_videos=50, solo_relevantes=True, jornada_filtro=JORNADA_OBJETIVO)
    todos_videos.extend(videos)

print(f"📹 Total videos a procesar: {len(todos_videos)}\n")

# --- Test previo con un solo video ---
print("🧪 Test con primer video...\n")
test = extraer_transcripcion(todos_videos[0])
if test["estado"] == "ok":
    print(f"✅ Proxy funcionando — {len(test['transcripcion_cruda']):,} caracteres")
    print(f"Primeros 200 chars: {test['transcripcion_cruda'][:200]}\n")

    # Si el test va bien, extraer todos
    print("🚀 Extrayendo todas las transcripciones...\n")
    videos_con_transcripcion = extraer_todas_transcripciones(todos_videos)

    # Guardar en JSON local
    with open("transcripciones_j38.json", "w", encoding="utf-8") as f:
        json.dump(videos_con_transcripcion, f, ensure_ascii=False, indent=2)

    ok    = sum(1 for v in videos_con_transcripcion if v["estado"] == "ok")
    error = sum(1 for v in videos_con_transcripcion if v["estado"] != "ok")
    print(f"\n📊 Resumen: {ok} ✅ con transcripción — {error} ❌ sin transcripción")
    print(f"💾 Guardadas en: transcripciones_j38.json")

else:
    print(f"❌ Proxy no funciona: {test['estado']}")
    print("Revisa tus credenciales de Webshare antes de continuar.")

📹 Total videos a procesar: 17

🧪 Test con primer video...

✅ Proxy funcionando — 103,902 caracteres
Primeros 200 chars: ¿A dónde vamos? Carra a circular. Carrasco Fantasy. El Fantasía es conocido mundialmente por estafar. Recomienda jugadores para ganar. No siempre acierta sus predicciones, pero la intención es la que 

🚀 Extrayendo todas las transcripciones...

  [1/17] carrasco — LA PREVIA FANTASY DE LA JORNADA 38 CON TODAS LAS ALINEACIONE...
           ✅ 103,902 caracteres
  [2/17] carrasco — 🔝 TOP 10 CAPITANES para la JORNADA 38 en LaLiga Fantasy 25/2...
           ✅ 1,108 caracteres
  [3/17] carrasco — MIS TIER LIST FANTASY POR POSICIONES PARA LA JORNADA 38...
           ❌ error: VideoUnplayable: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=3
  [4/17] carrasco — Mis ALINEACIONES PROBABLES y PRONÓSTICOS para la jornada 38 ...
           ✅ 33,441 caracteres
  [5/17] carrasco — 🚀 3 JUGADORES que harán más de 10 PUNTOS en la JORNADA 38 de...
         

In [15]:
# Reintentar transcripciones fallidas

fallidos = [v for v in videos_con_transcripcion if v["estado"] != "ok"]
print(f"🔄 Reintentando {len(fallidos)} videos fallidos...\n")

for i, video in enumerate(fallidos, 1):
    print(f"  [{i}/{len(fallidos)}] {video['titulo'][:60]}...")
    resultado = extraer_transcripcion(video)

    if resultado["estado"] == "ok":
        chars = len(resultado["transcripcion_cruda"])
        print(f"           ✅ {chars:,} caracteres")
        # Actualizar en la lista principal
        for j, v in enumerate(videos_con_transcripcion):
            if v["video_id"] == video["video_id"]:
                videos_con_transcripcion[j] = resultado
                break
    else:
        print(f"           ❌ {resultado['estado']}")

# Guardar JSON actualizado
with open("transcripciones_j38.json", "w", encoding="utf-8") as f:
    json.dump(videos_con_transcripcion, f, ensure_ascii=False, indent=2)

ok    = sum(1 for v in videos_con_transcripcion if v["estado"] == "ok")
error = sum(1 for v in videos_con_transcripcion if v["estado"] != "ok")
print(f"\n📊 Resumen actualizado: {ok} ✅ — {error} ❌")
print(f"💾 JSON actualizado: transcripciones_j38.json")

🔄 Reintentando 2 videos fallidos...

  [1/2] ANÁLISIS FANTASY de la JORNADA 38 con ALINEACIONES PROBABLES...
           ✅ 38,257 caracteres
  [2/2] 🎙 PODCAST J38 PREVIA FANTASY 💥 ALINEACIONES, DUDAS y las APU...
           ✅ 55,985 caracteres

📊 Resumen actualizado: 17 ✅ — 0 ❌
💾 JSON actualizado: transcripciones_j38.json


## 🧹 Bloque 5 — Limpieza y control de calidad

Procesamos las transcripciones en crudo para dejarlas listas para enviar a OpenAI.

**Pasos:**
1. **Limpieza**: eliminamos etiquetas `[Música]`, `[Aplausos]`, caracteres extraños y espacios múltiples
2. **Control de calidad**: clasificamos cada transcripción según su longitud:
   - `< 500 chars` → descartada (transcripción rota o vacía)
   - `500 - 30.000 chars` → se procesa directamente
   - `> 30.000 chars` → se divide en chunks
3. **Chunking** — dividimos los textos largos en fragmentos de `10.000 chars` con `500 chars` de solapamiento para no perder contexto entre chunks.

**Configuración ajustable:**

| Parámetro | Valor | Descripción |
|---|---|---|
| `MIN_CHARS` | 500 | Mínimo para no descartar |
| `MAX_CHARS` | 30.000 | Máximo antes de chunkear |
| `CHUNK_SIZE` | 10.000 | Tamaño de cada chunk |
| `CHUNK_OVERLAP` | 500 | Solapamiento entre chunks |

> El output de este bloque es una lista plana de textos listos para OpenAI, donde cada elemento conoce su canal, video y jornada de origen.

In [5]:
# --- Configuración ---
MIN_CHARS    = 500
MAX_CHARS    = 30_000
CHUNK_SIZE   = 10_000
CHUNK_OVERLAP = 500

def limpiar_texto(texto: str) -> str:
    """Limpia artefactos típicos de transcripciones automáticas de YouTube."""
    # Eliminar etiquetas tipo [Música], [Aplausos], [Risas]
    texto = re.sub(r'\[.*?\]', '', texto)
    # Eliminar caracteres especiales extraños manteniendo puntuación española
    texto = re.sub(r'[^\w\s\.,;:¿?¡!áéíóúüñÁÉÍÓÚÜÑ]', ' ', texto)
    # Colapsar espacios múltiples y saltos de línea
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()


def chunkear_texto(texto: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list:
    """Divide texto largo en chunks con solapamiento para no perder contexto."""
    chunks = []
    start  = 0
    while start < len(texto):
        end = start + chunk_size
        chunks.append(texto[start:end])
        start = end - overlap  # Retrocedemos overlap para solapar contexto
    return chunks


def procesar_calidad(video: dict) -> list:
    """
    Limpia la transcripción y aplica control de calidad.
    Retorna lista de dicts listos para OpenAI:
    - 1 elemento si el texto cabe entero
    - N elementos si necesita chunkeo
    - 0 elementos si es descartado
    """
    # Sin transcripción
    if video["estado"] != "ok" or not video["transcripcion_cruda"]:
        return []

    texto_limpio = limpiar_texto(video["transcripcion_cruda"])
    chars        = len(texto_limpio)

    # Demasiado corta
    if chars < MIN_CHARS:
        print(f"  ⚠️  Descartado (muy corta: {chars} chars): {video['titulo'][:50]}")
        return []

    base = {
        "canal":   video["canal"],
        "titulo":  video["titulo"],
        "fecha":   video["fecha"],
        "jornada": video["jornada"],
        "url":     video["url"],
    }

    # Longitud correcta: procesar entero
    if chars <= MAX_CHARS:
        return [{**base, "texto": texto_limpio, "chunk": None, "total_chunks": 1}]

    # Demasiado larga: chunkear
    chunks = chunkear_texto(texto_limpio)
    print(f"  ✂️  Chunkeado en {len(chunks)} partes: {video['titulo'][:50]}")
    return [
        {**base, "texto": chunk, "chunk": i + 1, "total_chunks": len(chunks)}
        for i, chunk in enumerate(chunks)
    ]


def preparar_textos_para_llm(videos_con_transcripcion: list) -> list:
    """Aplica limpieza y control de calidad a todos los videos."""
    textos_listos = []
    total = len(videos_con_transcripcion)

    for i, video in enumerate(videos_con_transcripcion, 1):
        titulo_corto = video['titulo'][:55]
        resultados   = procesar_calidad(video)

        if not resultados:
            if video["estado"] == "ok":
                pass  # Ya imprime el warning dentro de procesar_calidad
            else:
                print(f"  [{i}/{total}] ❌ Sin transcripción: {titulo_corto}")
            continue

        chars = len(resultados[0]["texto"])
        chunks_str = f"{resultados[0]['total_chunks']} chunk(s)"
        print(f"  [{i}/{total}] ✅ {chars:,} chars — {chunks_str} — {titulo_corto}")
        textos_listos.extend(resultados)

    return textos_listos


# --- TEST ---
print("🧹 Limpiando y clasificando transcripciones...\n")
textos_para_llm = preparar_textos_para_llm(videos_con_transcripcion)

print(f"\n📊 Resumen:")
print(f"  Videos procesados : {len(videos_con_transcripcion)}")
print(f"  Textos listos     : {len(textos_para_llm)}")
print(f"\nEjemplo — primeros 300 chars del primer texto:")
print(textos_para_llm[0]["texto"][:300] if textos_para_llm else "Sin datos")

🧹 Limpiando y clasificando transcripciones...

  ✂️  Chunkeado en 11 partes: LA PREVIA FANTASY DE LA JORNADA 38 CON TODAS LAS A
  [1/17] ✅ 10,000 chars — 11 chunk(s) — LA PREVIA FANTASY DE LA JORNADA 38 CON TODAS LAS ALINEA
  [2/17] ✅ 1,108 chars — 1 chunk(s) — 🔝 TOP 10 CAPITANES para la JORNADA 38 en LaLiga Fantasy
  ✂️  Chunkeado en 7 partes: MIS TIER LIST FANTASY POR POSICIONES PARA LA JORNA
  [3/17] ✅ 10,000 chars — 7 chunk(s) — MIS TIER LIST FANTASY POR POSICIONES PARA LA JORNADA 38
  ✂️  Chunkeado en 4 partes: Mis ALINEACIONES PROBABLES y PRONÓSTICOS para la j
  [4/17] ✅ 10,000 chars — 4 chunk(s) — Mis ALINEACIONES PROBABLES y PRONÓSTICOS para la jornad
  [5/17] ✅ 2,264 chars — 1 chunk(s) — 🚀 3 JUGADORES que harán más de 10 PUNTOS en la JORNADA 
  ✂️  Chunkeado en 4 partes: El MEJOR EQUIPO para la JORNADA 38 de LaLiga Fanta
  [6/17] ✅ 10,000 chars — 4 chunk(s) — El MEJOR EQUIPO para la JORNADA 38 de LaLiga Fantasy SO
  [7/17] ✅ 1,097 chars — 1 chunk(s) — 🔒 Los 7 CLAUSULAZOS Uber 

In [6]:
# Vista previa de lo que recibirá el Bloque 6
print("👀 Vista previa de chunks para OpenAI:\n")
for i, t in enumerate(textos_para_llm, 1):
    chunk_str = f"chunk {t['chunk']}/{t['total_chunks']}" if t['total_chunks'] > 1 else "completo"
    print(f"  [{i}] {t['canal']} | {t['fecha']} | J{t['jornada']} | {chunk_str}")
    print(f"       Título : {t['titulo'][:60]}")
    print(f"       Chars  : {len(t['texto']):,}")
    print(f"       Texto  : {t['texto'][:150]}...")
    print()

👀 Vista previa de chunks para OpenAI:

  [1] carrasco | 2026-05-23 | J38 | chunk 1/11
       Título : LA PREVIA FANTASY DE LA JORNADA 38 CON TODAS LAS ALINEACIONE
       Chars  : 10,000
       Texto  : ¿A dónde vamos? Carra a circular. Carrasco Fantasy. El Fantasía es conocido mundialmente por estafar. Recomienda jugadores para ganar. No siempre acie...

  [2] carrasco | 2026-05-23 | J38 | chunk 2/11
       Título : LA PREVIA FANTASY DE LA JORNADA 38 CON TODAS LAS ALINEACIONE
       Chars  : 10,000
       Texto  : e temporada, aunque creo que no será el último vídeo porque hay que reaccionar a las predicciones de inicio de temporada. Cuando hablo de predicciones...

  [3] carrasco | 2026-05-23 | J38 | chunk 3/11
       Título : LA PREVIA FANTASY DE LA JORNADA 38 CON TODAS LAS ALINEACIONE
       Chars  : 10,000
       Texto  : co del Fantasy y lo voy a echar de menos. Pues es bueno que lo echis el de menos porque así significa que los vais a cuidar en vuestras ligas, la verd...

  [4] c

## 🤖 Bloque 6 — Extracción de insights con OpenAI

Procesamos cada chunk de transcripción con OpenAI para extraer insights estructurados sobre jugadores.

**Flujo:**
1. Si `insights_j38.json` existe en disco → carga directamente sin llamar a OpenAI
2. Si no existe → procesa todos los chunks y guarda el JSON

**Para cada chunk, OpenAI extrae:**
- Jugador y posición
- Acción recomendada — `comprar`, `vender`, `alinear` o `evitar`
- Confianza (0.0 - 1.0) — basada en el tono del analista
- Razón corta y criterios de análisis
- Quote original del analista si existe
- Flag `nombre_dudoso`: indica si el nombre puede estar mal transcrito

**Decisiones técnicas:**
- Modelo `gpt-5.4-nano`: buena relación calidad/coste para extracción estructurada
- `temperature=0.1`: respuestas consistentes y predecibles
- **Structured outputs**: el formato JSON se garantiza mediante un schema (`json_schema`), no por instrucciones en el prompt. Elimina por completo los errores de parseo
- Diccionario `ALIASES_JUGADORES` como respaldo para apodos conocidos

> 💾 **Archivo generado:** `insights_j38.json`. Descárgalo y consérvalo entre sesiones. Es el dato más importante del proyecto.

**Resultado:** ~577 insights estructurados extraídos de 17 videos de la Jornada 38.

In [7]:
# En caso de no tener los insights generados

# Diccionario de aliases conocidos
ALIASES_JUGADORES = {
    "la mine": "Lamine Yamal",
    "cuchito": "Cucho Hernández",
    "patecis": "Pathé Ciss",
    # Añadir más según se detecten errores
}

# --- Schema para structured outputs ---
INSIGHT_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "insights_extraidos",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "insights": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "jugador":        {"type": "string"},
                            "posicion":       {"type": "string", "enum": ["portero", "defensa", "centrocampista", "delantero"]},
                            "accion":         {"type": "string", "enum": ["comprar", "vender", "alinear", "evitar"]},
                            "confianza":      {"type": "number"},
                            "razon_corta":    {"type": "string"},
                            "nombre_dudoso":  {"type": "boolean"},
                            "criterios": {
                                "type": "object",
                                "properties": {
                                    "forma_actual":      {"type": ["string", "null"]},
                                    "dinamica_equipo":   {"type": ["string", "null"]},
                                    "calendario_rival":  {"type": ["string", "null"]},
                                    "prediccion_puntos": {"type": ["string", "null"]}
                                },
                                "required": ["forma_actual", "dinamica_equipo", "calendario_rival", "prediccion_puntos"],
                                "additionalProperties": False
                            },
                            "quote_original": {"type": ["string", "null"]}
                        },
                        "required": ["jugador", "posicion", "accion", "confianza", "razon_corta", "nombre_dudoso", "criterios", "quote_original"],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["insights"],
            "additionalProperties": False
        }
    }
}

PROMPT_EXTRACCION = """Eres un experto analista de fantasy football español.
Se te proporciona la transcripción de un video de YouTube de un canal de análisis fantasy de LaLiga.

CONTEXTO DEL VIDEO:
- Canal: {canal}
- Título: {titulo}
- Fecha: {fecha}
- Jornada: {jornada}

TRANSCRIPCIÓN:
{texto}

INSTRUCCIONES:
1. Extrae TODOS los insights sobre jugadores específicos que contengan una recomendación clara.
2. Los analistas usan apodos y jerga — normaliza SIEMPRE al nombre oficial del jugador.
3. Solo incluye jugadores con recomendación explícita: comprar, vender, alinear o evitar.
4. La confianza (0.0-1.0) debe reflejar cuán seguro suena el analista.
5. Si el analista duda, baja la confianza. Si es tajante, súbela.
6. Si el nombre de un jugador parece mal transcrito o no lo reconoces con certeza,
   añade "nombre_dudoso": true. Si estás seguro del nombre, "nombre_dudoso": false.

ALIASES CONOCIDOS (usa estos como referencia):
{aliases}

Devuelve ÚNICAMENTE un array JSON válido con esta estructura, sin texto adicional, sin markdown:
[
  {{
    "jugador": "Nombre Oficial Completo",
    "nombre_dudoso": true|false,
    "posicion": "portero|defensa|centrocampista|delantero",
    "accion": "comprar|vender|alinear|evitar",
    "confianza": 0.0,
    "razon_corta": "resumen en 1-2 líneas",
    "criterios": {{
      "forma_actual": "descripción o null",
      "dinamica_equipo": "descripción o null",
      "calendario_rival": "descripción o null",
      "prediccion_puntos": "descripción o null"
    }},
    "quote_original": "cita textual del analista si existe, o null"
  }}
]

Si no hay insights claros devuelve: []"""


In [8]:
# Encoder de gpt-4o
enc = tiktoken.encoding_for_model("gpt-4o-mini")

# Calcular tokens del prompt base (sin transcripción)
prompt_base = PROMPT_EXTRACCION.format(
    canal="carrasco",
    titulo="Test",
    fecha="2026-05-23",
    jornada=38,
    texto="",
    aliases=json.dumps(ALIASES_JUGADORES, ensure_ascii=False)
)
tokens_prompt_base = len(enc.encode(prompt_base))

# Calcular tokens de cada chunk
total_tokens_input  = 0
total_tokens_output = 0  # Estimado: ~500 tokens de output por chunk

print(f"📊 Análisis de tokens por chunk:\n")
print(f"  {'#':<4} {'Canal':<20} {'Chars':>8} {'Tokens texto':>14} {'Tokens total':>13}")
print(f"  {'-'*65}")

for i, chunk in enumerate(textos_para_llm, 1):
    tokens_texto = len(enc.encode(chunk["texto"]))
    tokens_total = tokens_prompt_base + tokens_texto
    total_tokens_input += tokens_total
    print(f"  {i:<4} {chunk['canal']:<20} {len(chunk['texto']):>8,} {tokens_texto:>14,} {tokens_total:>13,}")

total_tokens_output = len(textos_para_llm) * 500  # Estimado output

# Precios gpt-4o-mini (junio 2026)
precio_input  = (total_tokens_input  / 1_000_000) * 0.20
precio_output = (total_tokens_output / 1_000_000) * 1.25
precio_total  = precio_input + precio_output

print(f"\n📊 Resumen:")
print(f"  Chunks totales       : {len(textos_para_llm)}")
print(f"  Tokens prompt base   : {tokens_prompt_base:,}")
print(f"  Tokens input total   : {total_tokens_input:,}")
print(f"  Tokens output est.   : {total_tokens_output:,}")
print(f"\n💰 Coste estimado:")
print(f"  Input  : ${precio_input:.4f}")
print(f"  Output : ${precio_output:.4f}")
print(f"  TOTAL  : ${precio_total:.4f}")

📊 Análisis de tokens por chunk:

  #    Canal                   Chars   Tokens texto  Tokens total
  -----------------------------------------------------------------
  1    carrasco               10,000          2,418         2,888
  2    carrasco               10,000          2,421         2,891
  3    carrasco               10,000          2,472         2,942
  4    carrasco               10,000          2,537         3,007
  5    carrasco               10,000          2,496         2,966
  6    carrasco               10,000          2,544         3,014
  7    carrasco               10,000          2,777         3,247
  8    carrasco               10,000          2,796         3,266
  9    carrasco               10,000          2,712         3,182
  10   carrasco               10,000          2,778         3,248
  11   carrasco                8,677          2,393         2,863
  12   carrasco                1,108            295           765
  13   carrasco               10,000     

In [9]:
def extraer_insights_openai(texto_chunk: dict) -> list:
    """
    Llama a OpenAI con structured outputs para extraer insights en JSON.
    No requiere parseo manual ni reintentos por formato inválido.
    """
    prompt = PROMPT_EXTRACCION.format(
        canal=texto_chunk["canal"],
        titulo=texto_chunk["titulo"],
        fecha=texto_chunk["fecha"],
        jornada=texto_chunk["jornada"],
        texto=texto_chunk["texto"],
        aliases=json.dumps(ALIASES_JUGADORES, ensure_ascii=False)
    )

    response = client.chat.completions.create(
        model           = "gpt-5.4-nano",
        max_tokens      = 4000,
        temperature     = 0.1,
        messages        = [{"role": "user", "content": prompt}],
        response_format = INSIGHT_SCHEMA
    )

    contenido = response.choices[0].message.content
    data      = json.loads(contenido)
    insights  = data["insights"]

    # Enriquecer con metadata del video
    for insight in insights:
        insight["fuente"] = {
            "canal":   texto_chunk["canal"],
            "titulo":  texto_chunk["titulo"],
            "fecha":   texto_chunk["fecha"],
            "jornada": texto_chunk["jornada"],
            "url":     texto_chunk["url"],
            "chunk":   texto_chunk["chunk"]
        }

    return insights

def construir_base_conocimiento(textos_para_llm: list) -> list:
    """Itera todos los chunks y extrae insights con OpenAI."""
    todos_insights = []
    total = len(textos_para_llm)

    for i, chunk in enumerate(textos_para_llm, 1):
        chunk_str = f"chunk {chunk['chunk']}/{chunk['total_chunks']}" if chunk['total_chunks'] > 1 else "completo"
        print(f"  [{i}/{total}] {chunk['canal']} | J{chunk['jornada']} | {chunk_str}")
        print(f"           {chunk['titulo'][:60]}")

        insights = extraer_insights_openai(chunk)
        todos_insights.extend(insights)
        print(f"           ✅ {len(insights)} insights extraídos")

    return todos_insights

In [10]:
# TEST con solo el primer chunk
print("🤖 Test con primer chunk...\n")

chunk_test = textos_para_llm[0]
print(f"📹 Video : {chunk_test['titulo'][:70]}")
print(f"📦 Chunk : {chunk_test['chunk']}/{chunk_test['total_chunks']}")
print(f"📝 Chars : {len(chunk_test['texto']):,}\n")

insights_test = extraer_insights_openai(chunk_test)

print(f"📊 Insights extraídos: {len(insights_test)}\n")
for insight in insights_test:
    dudoso_str = " ⚠️ NOMBRE DUDOSO" if insight.get("nombre_dudoso") else ""
    print(f"  {insight['jugador']} ({insight['posicion']}) → {insight['accion'].upper()} | confianza: {insight['confianza']}{dudoso_str}")
    print(f"  Razón: {insight['razon_corta']}")
    print()

🤖 Test con primer chunk...

📹 Video : LA PREVIA FANTASY DE LA JORNADA 38 CON TODAS LAS ALINEACIONES CONFIRMA
📦 Chunk : 1/11
📝 Chars : 10,000

📊 Insights extraídos: 0



In [11]:
# TEST con solo el segundo chunk
print("🤖 Test con segundo chunk...\n")

chunk_test = textos_para_llm[1]
print(f"📹 Video : {chunk_test['titulo'][:70]}")
print(f"📦 Chunk : {chunk_test['chunk']}/{chunk_test['total_chunks']}")
print(f"📝 Chars : {len(chunk_test['texto']):,}\n")

insights_test = extraer_insights_openai(chunk_test)

print(f"📊 Insights extraídos: {len(insights_test)}\n")
for insight in insights_test:
    dudoso_str = " ⚠️ NOMBRE DUDOSO" if insight.get("nombre_dudoso") else ""
    print(f"  {insight['jugador']} ({insight['posicion']}) → {insight['accion'].upper()} | confianza: {insight['confianza']}{dudoso_str}")
    print(f"  Razón: {insight['razon_corta']}")
    print()

🤖 Test con segundo chunk...

📹 Video : LA PREVIA FANTASY DE LA JORNADA 38 CON TODAS LAS ALINEACIONES CONFIRMA
📦 Chunk : 2/11
📝 Chars : 10,000

📊 Insights extraídos: 20

  Eric García (defensa) → COMPRAR | confianza: 0.72
  Razón: Lo marca como objetivo principal para la venta/tierris por posibles minutos.

  Ferran Torres (delantero) → COMPRAR | confianza: 0.7
  Razón: Recomendado como objetivo principal para la venta/tierris.

  Vinícius Júnior (delantero) → EVITAR | confianza: 0.78
  Razón: Dice que tiene toda la pinta de que se confirme su ausencia.

  Pedri (centrocampista) → EVITAR | confianza: 0.62
  Razón: Lo ve con mala pinta para ser titular (y lo menciona como posible no titular).

  Courtois (portero) → ALINEAR | confianza: 0.35 ⚠️ NOMBRE DUDOSO
  Razón: Lo menciona como debate entre porteros; sugiere que podría ser mejor por jugar en el Bernabéu.

  Radu (portero) → ALINEAR | confianza: 0.55 ⚠️ NOMBRE DUDOSO
  Razón: Se decanta por Radu para alinearlo por lo que se juega el

In [10]:
# --- Cargar desde JSON o regenerar ---
if os.path.exists("insights_j38.json"):
    print("📂 Cargando insights desde JSON local...")
    with open("insights_j38.json", "r", encoding="utf-8") as f:
        todos_insights = json.load(f)
    print(f"✅ {len(todos_insights)} insights cargados desde insights_j38.json")

else:
    print("⚠️  JSON no encontrado. Extrayendo insights con OpenAI...\n")
    todos_insights = construir_base_conocimiento(textos_para_llm)

    with open("insights_j38.json", "w", encoding="utf-8") as f:
        json.dump(todos_insights, f, ensure_ascii=False, indent=2)

    print(f"\n📊 Total insights extraídos: {len(todos_insights)}")
    print(f"💾 Guardados en: insights_j38.json")

🤖 Extrayendo insights con OpenAI...

  [1/73] carrasco | J38 | chunk 1/11
           LA PREVIA FANTASY DE LA JORNADA 38 CON TODAS LAS ALINEACIONE
           ✅ 0 insights extraídos
  [2/73] carrasco | J38 | chunk 2/11
           LA PREVIA FANTASY DE LA JORNADA 38 CON TODAS LAS ALINEACIONE
           ✅ 16 insights extraídos
  [3/73] carrasco | J38 | chunk 3/11
           LA PREVIA FANTASY DE LA JORNADA 38 CON TODAS LAS ALINEACIONE
           ✅ 14 insights extraídos
  [4/73] carrasco | J38 | chunk 4/11
           LA PREVIA FANTASY DE LA JORNADA 38 CON TODAS LAS ALINEACIONE
           ✅ 20 insights extraídos
  [5/73] carrasco | J38 | chunk 5/11
           LA PREVIA FANTASY DE LA JORNADA 38 CON TODAS LAS ALINEACIONE
           ✅ 13 insights extraídos
  [6/73] carrasco | J38 | chunk 6/11
           LA PREVIA FANTASY DE LA JORNADA 38 CON TODAS LAS ALINEACIONE
           ✅ 17 insights extraídos
  [7/73] carrasco | J38 | chunk 7/11
           LA PREVIA FANTASY DE LA JORNADA 38 CON TODAS LAS ALI

NameError: name 'insights_test' is not defined

In [11]:
# Relanzar chunks fallidos (0 insights extraídos)
chunks_fallidos = []

for i, chunk in enumerate(textos_para_llm):
    insights_chunk = [
        ins for ins in todos_insights
        if ins["fuente"]["url"]   == chunk["url"]
        and ins["fuente"]["chunk"] == chunk["chunk"]
    ]
    if len(insights_chunk) == 0:
        chunks_fallidos.append((i, chunk))

print(f"🔄 Chunks sin insights detectados: {len(chunks_fallidos)}\n")

🔄 Chunks sin insights detectados: 21



In [12]:
nuevos_insights = []
for i, chunk in chunks_fallidos:
    chunk_str = f"chunk {chunk['chunk']}/{chunk['total_chunks']}" if chunk['chunk'] else "completo"
    print(f"  [{i+1}/{len(textos_para_llm)}] {chunk['canal']} | {chunk_str}")
    print(f"           {chunk['titulo'][:60]}")

    insights = extraer_insights_openai(chunk)
    nuevos_insights.extend(insights)
    print(f"           ✅ {len(insights)} insights extraídos")

# Añadir nuevos insights a los existentes
todos_insights.extend(nuevos_insights)

# Guardar JSON actualizado
with open("insights_j38.json", "w", encoding="utf-8") as f:
    json.dump(todos_insights, f, ensure_ascii=False, indent=2)

print(f"\n📊 Insights previos    : {len(todos_insights) - len(nuevos_insights)}")
print(f"📊 Insights nuevos     : {len(nuevos_insights)}")
print(f"📊 Total insights final: {len(todos_insights)}")
print(f"💾 JSON actualizado    : insights_j38.json")

  [1/73] carrasco | chunk 1/11
           LA PREVIA FANTASY DE LA JORNADA 38 CON TODAS LAS ALINEACIONE
           ✅ 0 insights extraídos
  [9/73] carrasco | chunk 9/11
           LA PREVIA FANTASY DE LA JORNADA 38 CON TODAS LAS ALINEACIONE
           ✅ 14 insights extraídos
  [15/73] carrasco | chunk 3/7
           MIS TIER LIST FANTASY POR POSICIONES PARA LA JORNADA 38
    ⚠️  Error parseando JSON: Unterminated string starting at: line 395 column 17 (char 13219)
    Respuesta recibida: [
  {
    "jugador": "João Félix",
    "nombre_dudoso": true,
    "posicion": "delantero",
    "accion": "evitar",
    "confianza": 0.2,
    "razon_corta": "El analista menciona un jugador tipo 'Cuchi
           ✅ 0 insights extraídos
  [18/73] carrasco | chunk 6/7
           MIS TIER LIST FANTASY POR POSICIONES PARA LA JORNADA 38
    ⚠️  Error parseando JSON: Unterminated string starting at: line 409 column 5 (char 12922)
    Respuesta recibida: [
  {
    "jugador": "Iván Romero",
    "nombre_dudoso": 

## 🔍 Bloque 7 — RAG con ChromaDB

Indexamos los insights en una base de datos vectorial para poder hacer búsquedas semánticas.

**¿Por qué RAG?**
El RAG (Retrieval Augmented Generation) nos permite buscar los insights más relevantes para una pregunta sin tener que pasar todos los datos al modelo — más eficiente, más preciso y más económico.

**Flujo:**
1. Cada insight se convierte en un texto indexable con toda su información semántica
2. OpenAI genera un embedding (`text-embedding-3-small`) por cada texto
3. Los embeddings se almacenan en ChromaDB con similitud coseno
4. Ante una consulta, se buscan los `top_k` insights más similares semánticamente

**Decisiones técnicas:**
- Modelo `text-embedding-3-small`: mejor calidad que SentenceTransformers en español, coste muy bajo (~$0.01 para 577 insights)
- Similitud coseno: más adecuada que distancia euclidiana para comparar textos
- `TOP_K = 10`: recuperamos hasta 10 insights por consulta
- `filtrar_dudosos`: parámetro opcional para excluir nombres dudosos del retrieval

**Nota:** ChromaDB se inicializa en memoria: se pierde al cerrar la sesión. Al reiniciar, basta con ejecutar este bloque para reindexar desde `insights_j38.json` en ~30 segundos.

**Mejora futura:** complementar el RAG con una base de datos de grafo (Neo4j) para modelar relaciones entre jugadores, equipos y jornadas. Especialmente útil para consultas complejas como *"jugadores del Real Madrid con rival fácil esta jornada"*.

In [14]:
import chromadb

# --- Configuración ---
MODELO_EMBEDDINGS = "text-embedding-3-small"
TOP_K             = 10

def generar_embedding(texto: str) -> list:
    """Genera embedding de un texto usando OpenAI."""
    response = client.embeddings.create(
        model = MODELO_EMBEDDINGS,
        input = texto
    )
    return response.data[0].embedding


def generar_embeddings_batch(textos: list, batch_size: int = 100) -> list:
    """
    Genera embeddings en batches para no saturar la API.
    OpenAI permite hasta 2048 inputs por llamada pero usamos 100 por seguridad.
    """
    embeddings = []
    total      = len(textos)

    for i in range(0, total, batch_size):
        batch = textos[i:i + batch_size]
        print(f"  Batch {i//batch_size + 1}/{(total//batch_size) + 1} — {len(batch)} textos...")
        response = client.embeddings.create(
            model = MODELO_EMBEDDINGS,
            input = batch
        )
        embeddings.extend([e.embedding for e in response.data])

    return embeddings


# Inicializar ChromaDB
chroma_client = chromadb.Client()
coleccion     = chroma_client.get_or_create_collection(
    name     = "fantasy_insights",
    metadata = {"hnsw:space": "cosine"}
)
print("✅ ChromaDB inicializado")


def insight_a_texto(insight: dict) -> str:
    """Convierte un insight a texto indexable para el embedding."""
    criterios = insight.get("criterios", {})
    partes = [
        f"{insight['jugador']}, {insight.get('posicion', '')}.",
        f"Recomendación: {insight['accion']}.",
        f"Razón: {insight.get('razon_corta', '')}.",
        f"Forma actual: {criterios.get('forma_actual', '')}." if criterios.get('forma_actual') else "",
        f"Dinámica equipo: {criterios.get('dinamica_equipo', '')}." if criterios.get('dinamica_equipo') else "",
        f"Calendario rival: {criterios.get('calendario_rival', '')}." if criterios.get('calendario_rival') else "",
        f"Predicción puntos: {criterios.get('prediccion_puntos', '')}." if criterios.get('prediccion_puntos') else "",
        f"Fuente: {insight['fuente']['canal']}, Jornada {insight['fuente']['jornada']}.",
    ]
    return " ".join([p for p in partes if p])


def indexar_insights(insights: list) -> None:
    """Genera embeddings e indexa todos los insights en ChromaDB."""
    if not insights:
        print("⚠️  No hay insights para indexar.")
        return

    textos    = [insight_a_texto(i) for i in insights]
    ids       = [f"insight_{idx}" for idx in range(len(insights))]
    metadatas = [
        {
            "jugador":       i["jugador"],
            "posicion":      i.get("posicion", ""),
            "accion":        i["accion"],
            "confianza":     float(i["confianza"]),
            "canal":         i["fuente"]["canal"],
            "fecha":         i["fuente"]["fecha"],
            "jornada":       int(i["fuente"]["jornada"]) if i["fuente"]["jornada"] else 0,
            "nombre_dudoso": bool(i.get("nombre_dudoso", False)),
            "url":           i["fuente"]["url"],
        }
        for i in insights
    ]

    print(f"⏳ Generando embeddings para {len(textos)} insights...")
    embeddings = generar_embeddings_batch(textos)

    coleccion.add(
        ids        = ids,
        documents  = textos,
        embeddings = embeddings,
        metadatas  = metadatas
    )
    print(f"✅ {len(insights)} insights indexados en ChromaDB")


def consultar_rag(pregunta: str, top_k: int = TOP_K, filtrar_dudosos: bool = False) -> list:
    """Busca los insights más relevantes para una pregunta."""
    embedding_pregunta = generar_embedding(pregunta)

    where = {"nombre_dudoso": False} if filtrar_dudosos else None

    resultados = coleccion.query(
        query_embeddings = [embedding_pregunta],
        n_results        = top_k,
        where            = where
    )

    insights_relevantes = []
    for i in range(len(resultados["ids"][0])):
        insights_relevantes.append({
            "texto":     resultados["documents"][0][i],
            "metadata":  resultados["metadatas"][0][i],
            "distancia": resultados["distances"][0][i],
        })

    return insights_relevantes


# --- TEST ---
print(f"\n📦 Indexando {len(todos_insights)} insights...\n")
indexar_insights(todos_insights)

print("\n🔍 Test de consulta RAG:\n")
pregunta_test = "¿Debo alinear a Pedri esta jornada?"
resultados    = consultar_rag(pregunta_test, top_k=3)

print(f"Pregunta: '{pregunta_test}'")
print(f"RAG encontró {len(resultados)} documentos relevantes:\n")
for i, r in enumerate(resultados, 1):
    m = r["metadata"]
    print(f"  [{i}] {m['jugador']} → {m['accion'].upper()} | confianza: {m['confianza']} | {m['canal']}")
    print(f"       {r['texto'][:120]}...")
    print(f"       Similitud: {1 - r['distancia']:.2f}")
    print()

✅ ChromaDB inicializado

📦 Indexando 577 insights...

⏳ Generando embeddings para 577 insights...
  Batch 1/6 — 100 textos...
  Batch 2/6 — 100 textos...
  Batch 3/6 — 100 textos...
  Batch 4/6 — 100 textos...
  Batch 5/6 — 100 textos...
  Batch 6/6 — 77 textos...
✅ 577 insights indexados en ChromaDB

🔍 Test de consulta RAG:

Pregunta: '¿Debo alinear a Pedri esta jornada?'
RAG encontró 3 documentos relevantes:

  [1] Pedri González → EVITAR | confianza: 0.7 | jornada_perfecta
       Pedri González, centrocampista. Recomendación: evitar. Razón: El analista comenta que Pedri está fuera y que no tiene se...
       Similitud: 0.69

  [2] Pedri → EVITAR | confianza: 0.7 | jornada_perfecta
       Pedri, centrocampista. Recomendación: evitar. Razón: El analista afirma que Pedri está fuera y que no tiene sentido por ...
       Similitud: 0.68

  [3] Pedri → EVITAR | confianza: 0.5 | carrasco
       Pedri, centrocampista. Recomendación: evitar. Razón: Indica que tiene mala pinta de que sea titu

## ⚽ Bloque 8 — Agente Analista

Genera una recomendación de alineación óptima para la jornada basándose en los insights del RAG.

**Flujo en dos fases:**

**Fase 1 — Resumen individual por jugador:**
- Para cada jugador de la plantilla, consulta el RAG con `top_k=10`
- Filtra insights con confianza `> 0.4` para eliminar ruido
- Llamada individual a OpenAI por jugador → resumen compacto y estructurado
- Resultado: un resumen por jugador con acción recomendada y confianza media

**Fase 2 — Recomendación final:**
- Todos los resúmenes individuales se pasan en una única llamada a OpenAI
- El modelo recomienda el 11 óptimo teniendo en cuenta posiciones y confianzas
- Resultado: alineación recomendada con justificación por jugador

**¿Por qué dos fases?**
Evitamos el problema de *"lost in the middle"*: los modelos pierden precisión cuando el contexto es muy largo. Cada llamada individual tiene contexto pequeño y enfocado, y la llamada final recibe resúmenes compactos en lugar de insights en bruto.

**Configuración:**

| Parámetro | Valor | Descripción |
|---|---|---|
| `UMBRAL_CONFIANZA` | 0.4 | Mínimo para considerar un insight |
| `TOP_K_RAG` | 10 | Insights máximos por jugador |
| `temperature` | 0.1 / 0.2 | Baja para respuestas consistentes |

> Introduce tu plantilla real en `MI_PLANTILLA` antes de ejecutar este bloque.

**Mejora futura:** añadir un agente especializado en optimización de alineación que resuelva el problema como programación lineal. Maximizar puntos esperados dado un presupuesto y restricciones de posición.

In [16]:
# --- Configuración ---
UMBRAL_CONFIANZA = 0.4
TOP_K_RAG        = 10
JORNADA_OBJETIVO = 38

MI_PLANTILLA = [
    {"nombre": "Augusto Batalla",     "posicion": "portero"},
    {"nombre": "Abel Bretones",         "posicion": "defensa"},
    {"nombre": "Andrei Ratiu",    "posicion": "defensa"},
    {"nombre": "Carlos Romero",         "posicion": "defensa"},
    {"nombre": "David Affengruber", "posicion": "defensa"},
    {"nombre": "Javi Rodríguez", "posicion": "defensa"},
    {"nombre": "Léo Pétrot", "posicion": "defensa"},
    {"nombre": "Aleix Febas",          "posicion": "centrocampista"},
    {"nombre": "Chidera Ejuke",           "posicion": "centrocampista"},
    {"nombre": "Hugo Álvarez",          "posicion": "centrocampista"},
    {"nombre": "Ilaix Moriba",          "posicion": "centrocampista"},
    {"nombre": "Omar Mascarell",          "posicion": "centrocampista"},
    {"nombre": "Óscar Trejo",          "posicion": "centrocampista"},
    {"nombre": "Alemao",   "posicion": "delantero"},
    {"nombre": "Fede Viñas",          "posicion": "delantero"},
    {"nombre": "Gonzalo García",  "posicion": "delantero"},
    {"nombre": "Gonzalo García",  "posicion": "delantero"},
    {"nombre": "Goncalo Guedes",  "posicion": "delantero"},
    {"nombre": "Kike García",  "posicion": "delantero"},
    {"nombre": "Vedat Muriqi",  "posicion": "delantero"},
]

PROMPT_RESUMEN_JUGADOR = """Eres un experto analista de fantasy football español para LaLiga.

JUGADOR: {jugador} ({posicion})
JORNADA: {jornada}

INSIGHTS DE EXPERTOS SOBRE ESTE JUGADOR:
{insights}

Basándote SOLO en estos insights, genera un resumen conciso del jugador para esta jornada.

Devuelve ÚNICAMENTE un JSON válido sin markdown:
{{
  "jugador": "{jugador}",
  "posicion": "{posicion}",
  "confianza_media": 0.0,
  "accion_recomendada": "alinear|sentar|fichar|evitar",
  "resumen": "2-3 líneas máximo con lo más relevante",
  "fuentes": ["canal - título video"],
  "sin_datos": false
}}

Si no hay insights suficientes devuelve sin_datos: true y confianza_media: 0.0"""

PROMPT_RECOMENDACION_FINAL_PLANTILLA = """Eres un experto analista de fantasy football español para LaLiga.

JORNADA: {jornada}

RESÚMENES DE CADA JUGADOR DE LA PLANTILLA:
{resumenes}

Basándote en estos resúmenes, recomienda el 11 óptimo para esta jornada.
Ten en cuenta:
- Equilibrio por posiciones (1 portero, 4 defensas, 3-4 centrocampistas, 2-3 delanteros)
- Prioriza jugadores con mayor confianza
- Justifica los casos dudosos

Devuelve ÚNICAMENTE un JSON válido sin markdown:
{{
  "once_recomendado": [
    {{
      "jugador": "nombre",
      "posicion": "portero|defensa|centrocampista|delantero",
      "alinear": true,
      "confianza": 0.0,
      "justificacion": "razón breve"
    }}
  ],
  "advertencias": ["jugadores con dudas o sin datos"],
  "resumen": "resumen general en 2-3 líneas"
}}"""


def obtener_insights_jugador(nombre: str, top_k: int = TOP_K_RAG, umbral: float = UMBRAL_CONFIANZA) -> list:
    """
    Consulta el RAG y filtra por umbral de confianza.
    Retorna máximo top_k insights con confianza >= umbral.
    """
    pregunta   = f"¿Qué dicen los expertos sobre {nombre} en fantasy football?"
    resultados = consultar_rag(pregunta, top_k=top_k)

    # Filtrar por umbral de confianza
    filtrados = [r for r in resultados if r["metadata"]["confianza"] >= umbral]
    return filtrados


def resumir_jugador(jugador: dict, jornada: int = JORNADA_OBJETIVO) -> dict:
    """
    Llamada individual a OpenAI para resumir un jugador
    basándose en sus insights del RAG.
    """
    insights = obtener_insights_jugador(jugador["nombre"])

    # Formatear insights para el prompt
    if insights:
        insights_texto = "\n".join([
            f"- {r['metadata']['accion'].upper()} | confianza: {r['metadata']['confianza']} | {r['metadata']['canal']}\n  {r['texto'][:200]}"
            for r in insights
        ])
    else:
        insights_texto = "Sin insights disponibles para este jugador."

    prompt = PROMPT_RESUMEN_JUGADOR.format(
        jugador  = jugador["nombre"],
        posicion = jugador["posicion"],
        jornada  = jornada,
        insights = insights_texto
    )

    response = client.chat.completions.create(
        model       = "gpt-5.4-mini",
        max_completion_tokens  = 500,
        temperature = 0.1,
        messages    = [{"role": "user", "content": prompt}]
    )

    contenido = response.choices[0].message.content.strip()
    contenido = re.sub(r'^```json\s*', '', contenido)
    contenido = re.sub(r'\s*```$', '', contenido)

    try:
        resumen = json.loads(contenido)
        # Añadir precio si viene del mercado
        if "precio" in jugador:
            resumen["precio"] = jugador["precio"]
        print(f"  ✅ {jugador['nombre']} — {resumen.get('accion_recomendada', '?').upper()} | confianza: {resumen.get('confianza_media', 0)} | insights usados: {len(insights)}")
        return resumen
    except json.JSONDecodeError as e:
        print(f"  ⚠️  Error parseando JSON de {jugador['nombre']}: {e}")
        return {"jugador": jugador["nombre"], "posicion": jugador["posicion"], "sin_datos": True}


def analizar_plantilla(plantilla: list, jornada: int = JORNADA_OBJETIVO) -> dict:
    """
    Flujo completo de análisis de plantilla:
    1. Resumir cada jugador individualmente
    2. Llamada final con todos los resúmenes
    """
    print("🔍 Resumiendo jugadores de la plantilla...\n")
    resumenes = [resumir_jugador(j, jornada) for j in plantilla]

    print("\n🤖 Generando recomendación final de plantilla...")
    prompt = PROMPT_RECOMENDACION_FINAL_PLANTILLA.format(
        jornada  = jornada,
        resumenes = json.dumps(resumenes, ensure_ascii=False, indent=2)
    )

    response = client.chat.completions.create(
        model       = "gpt-5.4-mini",
        max_completion_tokens  = 2000,
        temperature = 0.2,
        messages    = [{"role": "user", "content": prompt}]
    )

    contenido = response.choices[0].message.content.strip()
    contenido = re.sub(r'^```json\s*', '', contenido)
    contenido = re.sub(r'\s*```$', '', contenido)

    try:
        return json.loads(contenido)
    except json.JSONDecodeError as e:
        print(f"⚠️  Error parseando JSON final: {e}")
        return {"error": contenido}

def mostrar_recomendacion_plantilla(resultado: dict) -> None:
    print("\n" + "="*60)
    print(f"🏆 ALINEACIÓN RECOMENDADA — J{JORNADA_OBJETIVO}")
    print("="*60)

    for jugador in resultado.get("once_recomendado", []):
        estado = "✅ ALINEAR" if jugador["alinear"] else "❌ SENTAR"
        print(f"\n  {estado} | {jugador['jugador']} ({jugador['posicion']})")
        print(f"  Confianza    : {jugador['confianza']}")
        print(f"  Justificación: {jugador['justificacion']}")

    if resultado.get("advertencias"):
        print("\n⚠️  ADVERTENCIAS:")
        for adv in resultado["advertencias"]:
            print(f"  - {adv}")

    print(f"\n📋 RESUMEN: {resultado.get('resumen', '')}")

# --- TEST ---
print("⚽ ANÁLISIS DE PLANTILLA\n")
resultado_plantilla = analizar_plantilla(MI_PLANTILLA)
mostrar_recomendacion_plantilla(resultado_plantilla)


⚽ ANÁLISIS DE PLANTILLA

🔍 Resumiendo jugadores de la plantilla...

  ✅ Augusto Batalla — EVITAR | confianza: 0.0 | insights usados: 6
  ✅ Abel Bretones — ALINEAR | confianza: 0.45 | insights usados: 3
  ✅ Andrei Ratiu — EVITAR | confianza: 0.63 | insights usados: 7
  ✅ Carlos Romero — ALINEAR | confianza: 0.61 | insights usados: 8
  ✅ David Affengruber — EVITAR | confianza: 0.0 | insights usados: 4
  ✅ Javi Rodríguez — FICHAR | confianza: 0.45 | insights usados: 7
  ✅ Léo Pétrot — EVITAR | confianza: 0.0 | insights usados: 8
  ✅ Aleix Febas — ALINEAR | confianza: 0.57 | insights usados: 3
  ✅ Chidera Ejuke — ALINEAR | confianza: 0.0 | insights usados: 5
  ✅ Hugo Álvarez — EVITAR | confianza: 0.0 | insights usados: 4
  ✅ Ilaix Moriba — ALINEAR | confianza: 0.0 | insights usados: 6
  ✅ Omar Mascarell — ALINEAR | confianza: 0.48 | insights usados: 8
  ✅ Óscar Trejo — EVITAR | confianza: 0.0 | insights usados: 7
  ✅ Alemao — EVITAR | confianza: 0.0 | insights usados: 8
  ✅ Fede Viñas — EV

## 💬 Bloque 9 — Chatbot con Gradio

Interfaz conversacional que permite hacer preguntas sobre jugadores y la plantilla recomendada.

**Arquitectura — Orquestador con Function Calling:**
El chatbot usa el patrón de orquestador con tools de OpenAI. Ante cada pregunta, el modelo decide automáticamente qué fuente consultar:

- `consultar_rag`: busca insights de analistas sobre un jugador concreto.
- `consultar_plantilla`: devuelve la recomendación de alineación ya generada.

**Flujo por tipo de pregunta:**

| Pregunta | Tools llamadas | Llamadas a OpenAI |
|---|---|---|
| "Hola" | Ninguna | 1 |
| "¿Alineo a Pedri?" | `consultar_rag` | 2 |
| "¿Tiene cabida Raphinha en mi plantilla?" | `consultar_rag` + `consultar_plantilla` | 2 |

**Decisiones técnicas:**
- Function calling nativo de OpenAI: más ligero que LangChain para este caso
- Soporte de múltiples tools por turno: el orquestador puede llamar varias en paralelo
- `share=True` en Gradio: genera enlace público desde Colab sin necesidad de ngrok

**Preguntas de ejemplo para la demo:**
- *"¿Debo alinear a Pedri en la jornada 38?"*
- *"¿Qué dicen los expertos sobre Lamine Yamal?"*
- *"¿Cuál es mi 11 recomendado para la jornada 38?"*
- *"¿Quién está mejor esta jornada, Mbappe o Ferran Torres?"*
- *"¿Es seguro poner a Lewandowski esta jornada?"*

**Mejoras futuras:**
- Migración a LangChain con memoria conversacional y agentes ReAct
- Nuevas tools: agente de estadísticas, agente de noticias y lesiones
- Caché de resultados RAG por jugador en sesión para reducir costes

> Ejecuta este bloque como último paso. Se generará un enlace público de Gradio para acceder al chatbot desde el navegador.

In [29]:
# --- Tools que el orquestador puede llamar ---
tools = [
    {
        "type": "function",
        "function": {
            "name": "consultar_rag",
            "description": "Consulta los insights de los analistas de YouTube sobre un jugador concreto. Úsala cuando el usuario pregunte por un jugador específico, su forma, si debe alinearlo, etc.",
            "parameters": {
                "type": "object",
                "properties": {
                    "jugador": {
                        "type": "string",
                        "description": "Nombre del jugador sobre el que consultar"
                    }
                },
                "required": ["jugador"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "consultar_plantilla",
            "description": "Consulta la recomendación de plantilla ya generada por el agente analista. Úsala cuando el usuario pregunte por su 11 recomendado, su alineación óptima, o quiera saber qué jugadores sentar.",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    }
]


def ejecutar_tool(nombre_tool: str, argumentos: dict) -> str:
    """Ejecuta la tool seleccionada por el orquestador y retorna el resultado como string."""
    if nombre_tool == "consultar_rag":
        jugador  = argumentos.get("jugador", "")
        insights = obtener_insights_jugador(jugador)

        if not insights:
            return f"No hay insights disponibles para {jugador} en J{JORNADA_OBJETIVO}."

        lineas = [f"Insights de expertos sobre {jugador} (J{JORNADA_OBJETIVO}):"]
        for r in insights:
            m = r["metadata"]
            lineas.append(f"- {m['accion'].upper()} | confianza: {m['confianza']} | {m['canal']}")
            lineas.append(f"  {r['texto'][:200]}")
        return "\n".join(lineas)

    elif nombre_tool == "consultar_plantilla":
        if not resultado_plantilla:
            return "No hay recomendación de plantilla disponible. Ejecuta primero el análisis de plantilla."
        return f"Recomendación de plantilla J{JORNADA_OBJETIVO}:\n{json.dumps(resultado_plantilla, ensure_ascii=False, indent=2)}"

    return "Tool no reconocida."


def orquestador(mensaje: str, historial: list) -> str:
    """
    Orquestador principal con function calling.
    Decide qué tool usar según la pregunta del usuario.
    """
    # Construir mensajes con historial
    mensajes = [
        {
            "role": "system",
            "content": f"""Eres un asistente experto en fantasy football español para LaLiga, Jornada {JORNADA_OBJETIVO}.
Tienes acceso a dos fuentes de información:
1. consultar_rag: insights de analistas de YouTube sobre jugadores concretos
2. consultar_plantilla: recomendación de alineación ya generada para el usuario

Usa siempre las tools disponibles antes de responder.
Responde en español, de forma concisa y útil para el usuario de fantasy."""
        }
    ]

    # Añadir historial
    for human, assistant in historial:
        mensajes.append({"role": "user",      "content": human})
        mensajes.append({"role": "assistant", "content": assistant})

    # Añadir mensaje actual
    mensajes.append({"role": "user", "content": mensaje})

    # Primera llamada — el orquestador decide qué tool usar
    response = client.chat.completions.create(
        model    = "gpt-5.4-mini",
        messages = mensajes,
        tools    = tools,
        tool_choice = "auto"
    )

    mensaje_respuesta = response.choices[0].message

    # Si el modelo quiere usar una tool
    if mensaje_respuesta.tool_calls:
      # Añadir respuesta del modelo al historial
      mensajes.append(mensaje_respuesta)

      # Ejecutar TODAS las tools que el modelo ha pedido
      for tool_call in mensaje_respuesta.tool_calls:
          nombre     = tool_call.function.name
          argumentos = json.loads(tool_call.function.arguments)

          print(f"  🔧 Tool llamada: {nombre} | args: {argumentos}")
          resultado_tool = ejecutar_tool(nombre, argumentos)

          # Añadir resultado de cada tool al historial
          mensajes.append({
              "role":         "tool",
              "tool_call_id": tool_call.id,
              "content":      resultado_tool
          })

      # Segunda llamada con todos los resultados de tools
      response_final = client.chat.completions.create(
          model    = "gpt-4o",
          messages = mensajes,
          tools    = tools,
      )
      return response_final.choices[0].message.content

    # Si responde directamente sin tool
    return mensaje_respuesta.content


# --- Interfaz Gradio ---
with gr.Blocks(title="Fantasy Copilot J38") as demo:
    gr.Markdown("""
    # ⚽ Fantasy Copilot — Jornada 38
    **Pregúntame sobre tu plantilla, jugadores concretos o tu alineación recomendada.**
    """)

    chatbot = gr.Chatbot(
        label  = "Fantasy Copilot",
        height = 500
    )
    msg   = gr.Textbox(
        placeholder = "Escribe tu pregunta sobre fantasy...",
        label       = "Tu pregunta"
    )
    clear = gr.Button("🗑️ Limpiar chat")

    def responder(mensaje, historial):
        historial = historial or []
        print(f"\n💬 Usuario: {mensaje}")
        respuesta = orquestador(mensaje, historial)
        print(f"🤖 Copiloto: {respuesta[:100]}...")
        historial.append((mensaje, respuesta))
        return "", historial

    msg.submit(responder,   [msg, chatbot], [msg, chatbot])
    clear.click(lambda: [], None, chatbot)

print("🚀 Lanzando Fantasy Copilot...")
demo.launch(share=True)  # share=True genera enlace público desde Colab

/tmp/ipykernel_2198/2779560441.py:138: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_2198/2779560441.py:138: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


🚀 Lanzando Fantasy Copilot...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d323fad5e2d1885f68.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
